# 03-2. GRU 모델링 — 기존 프로젝트의 비교 후보

03-1 LightGBM과 같은 순서로 기본 학습 → 역할별 선택 → 피처·파라미터 탐색 → 혼동행렬 → 운영 조건 분리 → 조건·최대피크 평가 → 시각화 → 저장 → 결론 → 추가실험을 진행합니다.

| 역할 | 학습 목표 | 선택 지표 |
|---|---|---|
| 수치예측 | 1주 전 기준선의 잔차를 L2로 학습 | MAE |
| 피크경보 | 1주 전 기준선의 잔차를 분위수 손실로 학습 | F1 |

- 학습은 1~6월, 선택은 7월 `primary_eligible`입니다.
- `history_clean_eligible`은 민감도 평가에만 사용합니다.
- 입력은 `01-2preprocessing.ipynb`가 만든 팀 공통 전처리만 사용하며, GRU는 24시간 순서와 4개 출력 채널을 유지합니다.
- 테스트 데이터는 읽지 않습니다. 7월에서 반복 선택한 결과는 월별 롤링 검증이 필요합니다.
- 새 산출물은 `results/gru_24h_team_preprocessing/`에 저장합니다.

**커널을 재시작하고 위에서부터 실행하세요.** 후보별 GRU를 여러 번 학습하므로 실행 시간이 깁니다.


In [ ]:
from pathlib import Path
import copy
import json
import os
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

ROOT = Path.cwd()
if not (ROOT / 'results/preprocessing_24h/config.json').exists():
    ROOT = ROOT.parent
PREP = ROOT / 'results/preprocessing_24h'
OUT = ROOT / 'results/gru_24h_team_preprocessing'
OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
device = torch.device('cpu')
torch.set_num_threads(min(8, os.cpu_count() or 1))
BASE_PARAMS = {
    'hidden_size': 32, 'num_layers': 2, 'dropout': 0.15,
    'bidirectional': True, 'batch_size': 64, 'learning_rate': 1e-3,
    'weight_decay': 1e-4, 'max_epochs': 100, 'patience': 10,
    'peak_weight': 1.0,
}
print('PyTorch:', torch.__version__, '| device:', device)
print('저장 위치:', OUT.resolve())


## 1. 공통 전처리 데이터 불러오기

01-2의 전처리 배열, 피크 기준, 검증 마스크를 사용합니다. 배열과 메타데이터의 순서·형태·유한값을 확인합니다.


In [ ]:
prep_config = json.loads((PREP / 'config.json').read_text(encoding='utf-8'))
SEQUENCE_NAMES = prep_config['sequence_features']
RECENT_NAMES = prep_config['recent_features']
PEAK_THRESHOLD = float(prep_config['peak_threshold'])

def load_arrays(split):
    with np.load(PREP / f'arrays_{split}.npz') as saved:
        return {name: saved[name].copy() for name in
                ['sequence', 'recent', 'y', 'baseline_w1', 'sample_id']}

train, validation = load_arrays('train'), load_arrays('validation')
meta_validation = pd.read_csv(
    PREP / 'metadata_validation.csv', index_col='sample_id',
    parse_dates=['origin', 'target_end'],
)
np.testing.assert_array_equal(validation['sample_id'], meta_validation.index.to_numpy())
primary_rows = meta_validation['primary_eligible'].astype(bool).to_numpy()
strict_rows = meta_validation['history_clean_eligible'].astype(bool).to_numpy()
assert primary_rows.any() and np.all(~strict_rows | primary_rows)

for arrays in [train, validation]:
    n = len(arrays['sample_id'])
    assert arrays['sequence'].shape == (n, 24, len(SEQUENCE_NAMES))
    assert arrays['recent'].shape == (n, len(RECENT_NAMES))
    assert arrays['y'].shape == arrays['baseline_w1'].shape == (n, 24, 4)
    for name in ['sequence', 'recent', 'y', 'baseline_w1']:
        assert np.isfinite(arrays[name]).all(), name

feature_sets = {'team_common_25': list(SEQUENCE_NAMES)}
assert len(SEQUENCE_NAMES) == 25
print('학습:', train['y'].shape, '| 검증:', validation['y'].shape)
print('primary:', primary_rows.sum(), '| history_clean:', strict_rows.sum(),
      '| 피크 기준:', PEAK_THRESHOLD)
display(pd.DataFrame([
    {'feature_set': name, 'sequence_features': len(names),
     'GRU_input_channels': len(names) + len(RECENT_NAMES)}
    for name, names in feature_sets.items()
]))


## 2. GRU 입력과 잔차 학습

24시간 sequence에 직전 시간 전력 4개를 반복 결합합니다. 입력 평균·표준편차와 잔차 스케일은 학습 데이터에서만 계산합니다.

양방향 GRU는 예측 시점에 알려진 향후 24시간 생산계획·달력과 과거 정보만 읽습니다. 미래 실제 전력은 입력하지 않습니다. 조기종료는 각 역할의 검증 손실로 수행하고, 후보 간 최종 선택만 MAE 또는 F1으로 합니다.


In [ ]:
def make_input(arrays, names):
    indices = [SEQUENCE_NAMES.index(name) for name in names]
    recent = np.repeat(arrays['recent'][:, None, :], 24, axis=1)
    return np.concatenate(
        [arrays['sequence'][:, :, indices], recent], axis=2
    ).astype(np.float32)

class GRURegressor(nn.Module):
    def __init__(self, input_features, hidden_size=32, num_layers=2,
                 dropout=0.15, bidirectional=True, scalar_output=False):
        super().__init__()
        self.scalar_output = scalar_output
        self.gru = nn.GRU(
            input_features, hidden_size, num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        width = hidden_size * (2 if bidirectional else 1)
        self.output = nn.Linear(width, 1 if scalar_output else 4)

    def forward(self, x):
        sequence, hidden = self.gru(x)
        if self.scalar_output:
            summary = (torch.cat([hidden[-2], hidden[-1]], dim=1)
                       if self.gru.bidirectional else hidden[-1])
            return self.output(summary).squeeze(-1)
        return self.output(sequence)

def element_loss(prediction, target, objective, alpha):
    error = target - prediction
    if objective == 'mse':
        return error.square()
    if objective == 'quantile':
        return torch.maximum(alpha * error, (alpha - 1) * error)
    raise ValueError(objective)

RUN_CACHE = {}  # 같은 입력·설정의 중복 학습 방지


In [ ]:
def fit_gru(names, role, params=None, objective='mse', alpha=0.9,
            train_mask=None, valid_mask=None, tag='unified'):
    params = {**BASE_PARAMS, **(params or {})}
    cache_key = (
        tuple(names), role, json.dumps(params, sort_keys=True),
        objective, float(alpha), tag,
    )
    if cache_key in RUN_CACHE:
        return RUN_CACHE[cache_key]

    torch.manual_seed(SEED)
    np.random.seed(SEED)
    started = time.perf_counter()
    xtr_raw, xva_raw = make_input(train, names), make_input(validation, names)
    x_mean = xtr_raw.mean(axis=(0, 1), keepdims=True)
    x_std = np.maximum(xtr_raw.std(axis=(0, 1), keepdims=True), 1e-6)
    xtr, xva = (xtr_raw - x_mean) / x_std, (xva_raw - x_mean) / x_std

    target_tr = train['y'] - train['baseline_w1']
    target_va = validation['y'] - validation['baseline_w1']
    tmask = (np.ones_like(target_tr, dtype=bool)
             if train_mask is None else np.asarray(train_mask, dtype=bool))
    vmask = (np.ones_like(target_va, dtype=bool)
             if valid_mask is None else np.asarray(valid_mask, dtype=bool))
    vmask = vmask & primary_rows[:, None, None]
    assert tmask.shape == target_tr.shape and vmask.shape == target_va.shape
    assert tmask.any() and vmask.any()

    target_mean = float(target_tr[tmask].mean())
    target_std = max(float(target_tr[tmask].std()), 1e-6)
    scaled_tr = ((target_tr - target_mean) / target_std).astype(np.float32)
    scaled_va = ((target_va - target_mean) / target_std).astype(np.float32)
    train_weights = tmask.astype(np.float32)
    if objective == 'mse':
        train_weights *= np.where(
            train['y'] >= PEAK_THRESHOLD, params['peak_weight'], 1.0
        ).astype(np.float32)

    active_rows = tmask.reshape(len(tmask), -1).any(axis=1)
    loader = DataLoader(
        TensorDataset(
            torch.from_numpy(xtr[active_rows]),
            torch.from_numpy(scaled_tr[active_rows]),
            torch.from_numpy(train_weights[active_rows]),
        ),
        batch_size=params['batch_size'], shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    architecture = {
        'input_features': xtr.shape[-1],
        **{key: params[key] for key in
           ['hidden_size', 'num_layers', 'dropout', 'bidirectional']},
        'scalar_output': False,
    }
    model = GRURegressor(**architecture).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=params['learning_rate'],
        weight_decay=params['weight_decay'],
    )
    valid_x = torch.from_numpy(xva).to(device)
    valid_target = torch.from_numpy(scaled_va).to(device)
    valid_mask_tensor = torch.from_numpy(vmask.astype(np.float32)).to(device)

    best_loss, best_epoch, best_state, stale = np.inf, 0, None, 0
    history = []
    print(f'학습 시작: {role} / {tag} / {xtr.shape[-1]}채널 / {objective}')
    for epoch in range(1, params['max_epochs'] + 1):
        model.train()
        train_sum, weight_sum = 0.0, 0.0
        for batch_x, batch_target, batch_weight in loader:
            batch_x, batch_target, batch_weight = [
                value.to(device) for value in (batch_x, batch_target, batch_weight)
            ]
            optimizer.zero_grad()
            raw = element_loss(model(batch_x), batch_target, objective, alpha)
            loss = (raw * batch_weight).sum() / batch_weight.sum()
            if not torch.isfinite(loss):
                raise RuntimeError('학습 손실이 유한하지 않습니다.')
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_sum += float(loss.detach()) * float(batch_weight.sum())
            weight_sum += float(batch_weight.sum())

        model.eval()
        with torch.no_grad():
            valid_raw = element_loss(model(valid_x), valid_target, objective, alpha)
            valid_loss = float(
                (valid_raw * valid_mask_tensor).sum() / valid_mask_tensor.sum()
            )
        history.append({
            'epoch': epoch, 'train_loss': train_sum / weight_sum,
            'valid_loss': valid_loss,
        })
        if valid_loss < best_loss - 1e-6:
            best_loss, best_epoch, stale = valid_loss, epoch, 0
            best_state = {
                name: tensor.detach().cpu().clone()
                for name, tensor in model.state_dict().items()
            }
        else:
            stale += 1
            if stale >= params['patience']:
                break

    model.load_state_dict(best_state)
    model.eval()
    parts = []
    with torch.no_grad():
        for start in range(0, len(xva), params['batch_size']):
            parts.append(model(
                torch.from_numpy(xva[start:start + params['batch_size']]).to(device)
            ).cpu().numpy())
    predicted_residual = np.concatenate(parts) * target_std + target_mean
    prediction = np.maximum(
        validation['baseline_w1'] + predicted_residual, 0
    ).astype(np.float32)
    run = {
        'names': list(names), 'params': params, 'objective': objective,
        'alpha': float(alpha), 'architecture': architecture,
        'state_dict': best_state, 'x_mean': x_mean, 'x_std': x_std,
        'target_mean': target_mean, 'target_std': target_std,
        'prediction': prediction, 'history': pd.DataFrame(history),
        'best_epoch': best_epoch, 'best_valid_loss': best_loss,
        'train_positions': int(tmask.sum()),
        'validation_positions': int(vmask.sum()),
        'seconds': time.perf_counter() - started,
    }
    RUN_CACHE[cache_key] = run
    print(f'완료: best epoch {best_epoch} / {run["seconds"]:.1f}초')
    return run


## 3. 공통 평가 함수

15분별 수치예측은 MAE·RMSE, 피크경보는 Precision·Recall·F1·오탐·미탐을 계산합니다. 혼동행렬의 수치는 겹치는 예측창의 15분 칸 수이며 고유 피크 사건 수가 아닙니다.


In [ ]:
def evaluate(actual, prediction, mask=None):
    actual, prediction = np.asarray(actual), np.asarray(prediction)
    if mask is not None:
        mask = np.broadcast_to(mask, actual.shape)
        actual, prediction = actual[mask], prediction[mask]
    actual, prediction = actual.reshape(-1), prediction.reshape(-1)
    error = prediction - actual
    actual_peak, predicted_peak = (
        actual >= PEAK_THRESHOLD, prediction >= PEAK_THRESHOLD
    )
    tp = int((actual_peak & predicted_peak).sum())
    fp = int((~actual_peak & predicted_peak).sum())
    fn = int((actual_peak & ~predicted_peak).sum())
    tn = int((~actual_peak & ~predicted_peak).sum())
    return {
        'observations': len(actual),
        'mae': float(np.abs(error).mean()) if len(error) else np.nan,
        'rmse': float(np.sqrt(np.mean(error ** 2))) if len(error) else np.nan,
        'mean_error': float(error.mean()) if len(error) else np.nan,
        'true_positive': tp, 'false_positive': fp,
        'false_negative': fn, 'true_negative': tn,
        'peak_precision': tp / (tp + fp) if tp + fp else np.nan,
        'peak_recall': tp / (tp + fn) if tp + fn else np.nan,
        'peak_f1': 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan,
        'peak_f2': 5 * tp / (5 * tp + fp + 4 * fn) if 5 * tp + fp + 4 * fn else np.nan,
    }

def maximum_metrics(actual, prediction):
    actual, prediction = np.asarray(actual), np.asarray(prediction)
    error = prediction - actual
    return {
        'windows': len(actual),
        'maximum_mae': float(np.abs(error).mean()) if len(error) else np.nan,
        'maximum_rmse': float(np.sqrt(np.mean(error ** 2))) if len(error) else np.nan,
        'maximum_bias': float(error.mean()) if len(error) else np.nan,
        'underprediction_rate': float((error < 0).mean()) if len(error) else np.nan,
    }

def window_metrics(actual, prediction):
    actual, prediction = actual.reshape(-1, 96), prediction.reshape(-1, 96)
    actual_max, predicted_max = actual.max(axis=1), prediction.max(axis=1)
    distance = (
        np.abs(np.arange(96)[None, :] - prediction.argmax(axis=1)[:, None]) * 15
    )
    timing = np.where(
        actual == actual_max[:, None], distance, np.inf
    ).min(axis=1)
    return {
        **maximum_metrics(actual_max, predicted_max),
        'maximum_time_mae_minutes': float(timing.mean()),
        'maximum_time_within_60m': float((timing <= 60).mean()),
    }

def result_table(runs):
    return pd.DataFrame([
        {
            'role': role, 'candidate': candidate,
            'channels': len(run['names']) + len(RECENT_NAMES),
            'best_epoch': run['best_epoch'],
            **evaluate(
                validation['y'][primary_rows],
                run['prediction'][primary_rows],
            ),
        }
        for (role, candidate), run in runs.items()
    ])

def show_roles(table, id_columns):
    for role, metrics in [
        ('point', ['mae', 'rmse']),
        ('peak_f1', [
            'peak_precision', 'peak_recall', 'peak_f1',
            'false_positive', 'false_negative',
        ]),
    ]:
        print('\n수치예측 (MAE 기준)' if role == 'point' else '\n피크경보 (F1 기준)')
        display(
            table.loc[table['role'].eq(role), id_columns + metrics].round(4)
        )

def select_id(table, role, column='candidate'):
    rows = table.loc[table['role'].eq(role)]
    if role == 'point':
        return rows.sort_values('mae', kind='stable').iloc[0][column]
    return rows.sort_values(
        ['peak_f1', 'peak_precision', 'mae'],
        ascending=[False, False, True], kind='stable',
    ).iloc[0][column]

def plot_confusions(predictions, path):
    fig, axes = plt.subplots(
        1, len(predictions), figsize=(6 * len(predictions), 4.8), squeeze=False
    )
    for ax, (label, prediction) in zip(axes.ravel(), predictions.items()):
        score = evaluate(
            validation['y'][primary_rows], prediction[primary_rows]
        )
        matrix = np.array([
            [score['true_negative'], score['false_positive']],
            [score['false_negative'], score['true_positive']],
        ])
        ax.imshow(matrix, cmap='Blues')
        for (row, col), value in np.ndenumerate(matrix):
            ax.text(
                col, row, f'{value:,}\n({value / matrix.sum():.2%})',
                ha='center', va='center',
                color='white' if value > matrix.max() / 2 else 'black',
            )
        ax.set(
            xticks=[0, 1], yticks=[0, 1],
            xticklabels=['예측 비피크', '예측 피크'],
            yticklabels=['실제 비피크', '실제 피크'],
            xlabel='예측', ylabel='실제',
            title=(
                f'{label}\nPrecision {score["peak_precision"]:.3f} / '
                f'Recall {score["peak_recall"]:.3f} / F1 {score["peak_f1"]:.3f}'
            ),
        )
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches='tight')
    plt.show()


## 4. 기본 GRU 학습

같은 `team_common_25` 입력으로 수치예측 L2 모델과 피크경보 분위수 0.90 모델을 학습합니다. 전주 동일 시점 전력은 공통 기준선으로 표시합니다.


In [ ]:
basic_runs = {
    ('point', 'L2'): fit_gru(
        feature_sets['team_common_25'], 'point', objective='mse'
    ),
    ('peak_f1', 'quantile_0.90'): fit_gru(
        feature_sets['team_common_25'], 'peak_f1',
        objective='quantile', alpha=0.90,
    ),
}
basic_results = result_table(basic_runs)
show_roles(basic_results, ['candidate', 'channels', 'best_epoch'])
display(pd.DataFrame([{
    'model': 'previous_week_baseline',
    **evaluate(
        validation['y'][primary_rows],
        validation['baseline_w1'][primary_rows],
    ),
}]).round(4))
basic_results.to_csv(
    OUT / 'basic_metrics_validation.csv',
    index=False, encoding='utf-8-sig',
)


## 5. 기본 역할별 선택

수치예측은 L2 후보, 피크경보는 분위수 후보를 역할별 출발점으로 사용합니다. 두 모델은 입력이 같아도 손실과 선택된 epoch가 다릅니다.


In [ ]:
basic_selected = {
    role: basic_runs[(role, select_id(basic_results, role))]
    for role in ['point', 'peak_f1']
}
for role, run in basic_selected.items():
    print(
        role, '| objective:', run['objective'],
        '| alpha:', run['alpha'], '| best epoch:', run['best_epoch']
    )


## 5-1. 피처 묶음 및 하이퍼파라미터 비교

01-2의 25개 공통 변수를 고정한 뒤 구조 → 학습 설정 → 역할별 손실 설정 순으로 탐색합니다. 각 단계는 직전 단계의 선택값을 고정한 제한된 탐색입니다.

- 구조: 현재 설정, hidden 64, 1층 GRU
- 학습: 현재 설정, 학습률 0.0005, batch 128
- 수치예측: 피크 표본 가중치 1·2·5
- 피크경보: 분위수 0.85·0.90·0.95

F2는 계산만 하고 선택에는 사용하지 않습니다.


In [ ]:
feature_runs = {}
for role, base in basic_selected.items():
    for candidate, names in feature_sets.items():
        feature_runs[(role, candidate)] = fit_gru(
            names, role, base['params'], base['objective'], base['alpha']
        )
feature_results = result_table(feature_runs)
show_roles(feature_results, ['candidate', 'channels', 'best_epoch'])
feature_results.to_csv(
    OUT / 'feature_metrics_validation.csv',
    index=False, encoding='utf-8-sig',
)
feature_selected = {
    role: feature_runs[(role, select_id(feature_results, role))]
    for role in basic_selected
}

def search_stage(stage, current, candidates):
    runs = {}
    for role, base in current.items():
        for candidate, changes in candidates.items():
            runs[(role, candidate)] = fit_gru(
                base['names'], role,
                {**base['params'], **changes},
                base['objective'], base['alpha'],
            )
    table = result_table(runs)
    show_roles(table, ['candidate', 'channels', 'best_epoch'])
    table.to_csv(
        OUT / f'{stage}_validation.csv',
        index=False, encoding='utf-8-sig',
    )
    selected = {
        role: runs[(role, select_id(table, role))]
        for role in current
    }
    return selected, table

structure_selected, structure_results = search_stage(
    'structure', feature_selected, {
        'current': {},
        'hidden64': {'hidden_size': 64},
        'one_layer': {'num_layers': 1, 'dropout': 0.0},
    }
)
training_selected, training_results = search_stage(
    'training', structure_selected, {
        'current': {},
        'lr0005': {'learning_rate': 5e-4},
        'batch128': {'batch_size': 128},
    }
)


In [ ]:
loss_runs = {}
point_base = training_selected['point']
for weight in [1.0, 2.0, 5.0]:
    loss_runs[('point', f'weight_{weight:g}')] = fit_gru(
        point_base['names'], 'point',
        {**point_base['params'], 'peak_weight': weight},
        objective='mse',
    )
risk_base = training_selected['peak_f1']
for alpha in [0.85, 0.90, 0.95]:
    loss_runs[('peak_f1', f'quantile_{alpha:.2f}')] = fit_gru(
        risk_base['names'], 'peak_f1', risk_base['params'],
        objective='quantile', alpha=alpha,
    )
loss_results = result_table(loss_runs)
show_roles(loss_results, ['candidate', 'channels', 'best_epoch'])
loss_results.to_csv(
    OUT / 'loss_settings_validation.csv',
    index=False, encoding='utf-8-sig',
)
tuned_selected = {
    role: loss_runs[(role, select_id(loss_results, role))]
    for role in basic_selected
}


## 5-2. 최적 조합 혼동행렬

MAE로 선택한 수치예측 모델과 F1으로 선택한 피크경보 모델을 각각 표시합니다. 수치예측 모델의 혼동행렬은 참고용이며, 실제 경보 성능은 피크경보 모델을 기준으로 봅니다.


In [ ]:
plot_confusions({
    '수치예측 최적 조합 (MAE 선택)': tuned_selected['point']['prediction'],
    '피크경보 최적 조합 (F1 선택)': tuned_selected['peak_f1']['prediction'],
}, OUT / 'tuned_confusions_validation.png')


## 5-3. 운영 조건 분리

통합 / 생산 여부 / 생산량 수준 / 평일과 주말·공휴일의 네 전략을 비교합니다. 저·고생산 경계는 학습 기간의 양수 생산계획 중앙값으로 고정합니다.

각 전문 GRU는 24시간 입력 전체를 읽고 해당 조건의 출력 위치에만 손실을 적용합니다. 예측도 대상 시간별로 결합합니다. 수치예측 전략과 피크경보 전략은 각각 MAE와 F1으로 독립 선택합니다.


In [ ]:
def sequence_feature(arrays, name):
    return arrays['sequence'][:, :, SEQUENCE_NAMES.index(name)]

positive_plans = sequence_feature(train, 'planned_production')
positive_plans = positive_plans[positive_plans > 0]
production_split = float(np.median(positive_plans))

def make_regime_masks(arrays):
    planned = sequence_feature(arrays, 'planned_production')
    day_off = (
        (sequence_feature(arrays, 'weekend') > 0) |
        (sequence_feature(arrays, 'holiday') > 0)
    )
    hour_masks = {
        'unified': {'all': np.ones_like(planned, dtype=bool)},
        'production_binary': {
            'production_off': planned <= 0,
            'production_on': planned > 0,
        },
        'production_level': {
            'production_off': planned <= 0,
            'production_low': (
                (planned > 0) & (planned <= production_split)
            ),
            'production_high': planned > production_split,
        },
        'calendar': {
            'weekday': ~day_off,
            'weekend_or_holiday': day_off,
        },
    }
    result = {
        strategy: {
            group: np.repeat(mask[:, :, None], 4, axis=2)
            for group, mask in groups.items()
        }
        for strategy, groups in hour_masks.items()
    }
    for groups in result.values():
        assert np.all(np.stack(list(groups.values())).sum(axis=0) == 1)
    return result

train_regimes = make_regime_masks(train)
validation_regimes = make_regime_masks(validation)

def fit_routed(role, reference, strategy, tag_prefix='regime'):
    if strategy == 'unified':
        return {
            'prediction': reference['prediction'],
            'models': {'all': reference},
            'fallbacks': {},
            'strategy': strategy,
        }
    prediction = np.full_like(validation['y'], np.nan, dtype=np.float32)
    models, fallbacks = {}, {}
    for group, train_mask in train_regimes[strategy].items():
        valid_mask = validation_regimes[strategy][group]
        eligible_valid = valid_mask & primary_rows[:, None, None]
        if not train_mask.any() or not eligible_valid.any():
            run = reference
            fallbacks[group] = '통합 모델 사용: 학습 또는 검증 위치 없음'
        else:
            run = fit_gru(
                reference['names'], role, reference['params'],
                reference['objective'], reference['alpha'],
                train_mask=train_mask, valid_mask=valid_mask,
                tag=f'{tag_prefix}/{strategy}/{group}',
            )
        models[group] = run
        prediction[valid_mask] = run['prediction'][valid_mask]
    assert np.isfinite(prediction).all()
    return {
        'prediction': prediction, 'models': models,
        'fallbacks': fallbacks, 'strategy': strategy,
    }

regime_runs, regime_rows, regime_group_rows = {}, [], []
for role, reference in tuned_selected.items():
    for strategy in train_regimes:
        routed = fit_routed(role, reference, strategy)
        regime_runs[(role, strategy)] = routed
        regime_rows.append({
            'role': role, 'strategy': strategy,
            **evaluate(
                validation['y'][primary_rows],
                routed['prediction'][primary_rows],
            ),
        })
        for group, mask in validation_regimes[strategy].items():
            regime_group_rows.append({
                'role': role, 'strategy': strategy, 'group': group,
                'fallback': routed['fallbacks'].get(group, ''),
                **evaluate(
                    validation['y'], routed['prediction'],
                    mask & primary_rows[:, None, None],
                ),
            })

regime_results = pd.DataFrame(regime_rows)
regime_group_results = pd.DataFrame(regime_group_rows)
selected_strategy = {
    role: select_id(regime_results, role, 'strategy')
    for role in tuned_selected
}
selected_routed = {
    role: regime_runs[(role, selected_strategy[role])]
    for role in tuned_selected
}
selected_predictions = {
    role: routed['prediction']
    for role, routed in selected_routed.items()
}
show_roles(regime_results, ['strategy'])
print('저/고생산 경계:', production_split)
print('역할별 선택:', selected_strategy)
display(regime_group_results.round(4))
regime_results.to_csv(
    OUT / 'regime_metrics_validation.csv',
    index=False, encoding='utf-8-sig',
)
regime_group_results.to_csv(
    OUT / 'regime_groups_validation.csv',
    index=False, encoding='utf-8-sig',
)


## 5-4. 조건·최대피크 평가

5-3에서 선택한 두 역할을 primary와 history_clean 구간에 적용합니다. 01-2에 포함된 현재와 전주의 생산 상태별로 평가합니다.

최대전력 크기와 발생 시각은 수치예측 곡선과 전주 기준선을 중심으로 봅니다. 경보용 분위수 곡선의 최대값은 참고용입니다.


In [ ]:
plan_on = sequence_feature(validation, 'planned_production') > 0
w1_on = sequence_feature(validation, 'w1_production') > 0
condition_hours = {
    'all': np.ones_like(plan_on, dtype=bool),
    'both_off': ~plan_on & ~w1_on,
    'current_only_production': plan_on & ~w1_on,
    'previous_week_only_production': ~plan_on & w1_on,
    'both_on': plan_on & w1_on,
}
conditions = {
    name: np.repeat(mask[:, :, None], 4, axis=2)
    for name, mask in condition_hours.items()
}
evaluation_rows = {'primary': primary_rows, 'history_clean': strict_rows}
diagnostic_predictions = {
    'baseline': validation['baseline_w1'],
    **selected_predictions,
}
condition_records, maximum_records = [], []
for subset, rows in evaluation_rows.items():
    for role, prediction in diagnostic_predictions.items():
        maximum_records.append({
            'subset': subset, 'role': role,
            **window_metrics(
                validation['y'][rows], prediction[rows]
            ),
        })
        for condition, mask in conditions.items():
            condition_records.append({
                'subset': subset, 'role': role,
                'condition': condition,
                **evaluate(
                    validation['y'], prediction,
                    mask & rows[:, None, None],
                ),
            })
condition_results = pd.DataFrame(condition_records)
maximum_results = pd.DataFrame(maximum_records)
for subset in evaluation_rows:
    print('\n', subset)
    show_roles(
        condition_results.loc[
            condition_results['subset'].eq(subset)
        ],
        ['condition', 'observations'],
    )
display(maximum_results.round(4))
condition_results.to_csv(
    OUT / 'conditions_validation.csv',
    index=False, encoding='utf-8-sig',
)
maximum_results.to_csv(
    OUT / 'maximum_validation.csv',
    index=False, encoding='utf-8-sig',
)


## 6. 시각화

단계별 성능, 조건 분리 전략, 최종 혼동행렬, 생산조건별 성능, 최대전력 오차·편향, 예측 사례를 표시합니다. MAE 그래프는 수치예측 모델, F1 그래프는 피크경보 모델의 결과입니다. 사례는 primary 구간에서 실제 최대전력이 가장 큰 창이며 대표 성능을 뜻하지 않습니다.


In [ ]:
stage_records = []
for stage, runs in [
    ('basic', basic_selected),
    ('features', feature_selected),
    ('structure', structure_selected),
    ('training', training_selected),
    ('loss_settings', tuned_selected),
]:
    for role, run in runs.items():
        stage_records.append({
            'stage': stage, 'role': role,
            **evaluate(
                validation['y'][primary_rows],
                run['prediction'][primary_rows],
            ),
        })
for role, prediction in selected_predictions.items():
    stage_records.append({
        'stage': 'regime', 'role': role,
        **evaluate(
            validation['y'][primary_rows],
            prediction[primary_rows],
        ),
    })
progress_results = pd.DataFrame(stage_records)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for col, (role, metric, title) in enumerate([
    ('point', 'mae', '수치예측 MAE'),
    ('peak_f1', 'peak_f1', '피크경보 F1'),
]):
    table = progress_results.loc[progress_results['role'].eq(role)]
    axes[0, col].plot(table['stage'], table[metric], marker='o')
    axes[0, col].set_title(title + ' / 단계별')
    axes[0, col].tick_params(axis='x', rotation=30)
    table = regime_results.loc[regime_results['role'].eq(role)]
    axes[1, col].bar(table['strategy'], table[metric])
    axes[1, col].set_title(title + ' / 운영 조건 분리')
    axes[1, col].tick_params(axis='x', rotation=25)
fig.tight_layout()
fig.savefig(
    OUT / 'progress_validation.png',
    dpi=160, bbox_inches='tight',
)
plt.show()

plot_confusions({
    '최종 수치예측 (MAE 선택)': selected_predictions['point'],
    '최종 피크경보 (F1 선택)': selected_predictions['peak_f1'],
}, OUT / 'selected_confusions_validation.png')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, role, metric, title in [
    (axes[0, 0], 'point', 'mae', '수치예측 조건별 MAE'),
    (axes[0, 1], 'peak_f1', 'peak_f1', '피크경보 조건별 F1'),
]:
    table = condition_results.loc[
        condition_results['subset'].eq('primary') &
        condition_results['role'].eq(role)
    ]
    ax.bar(table['condition'], table[metric])
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=35)

maximum_plot = maximum_results.loc[
    maximum_results['subset'].eq('primary') &
    maximum_results['role'].isin(['baseline', 'point'])
]
for ax, metric, title in [
    (axes[1, 0], 'maximum_mae', '24시간 최대전력 MAE'),
    (axes[1, 1], 'maximum_bias', '최대전력 편향: 음수는 과소예측'),
]:
    ax.bar(maximum_plot['role'], maximum_plot[metric])
    ax.set_title(title)
    ax.axhline(0, color='black', linewidth=0.8)
fig.tight_layout()
fig.savefig(
    OUT / 'conditions_and_maximum_validation.png',
    dpi=160, bbox_inches='tight',
)
plt.show()

eligible = np.flatnonzero(primary_rows)
example_index = eligible[
    np.argmax(validation['y'][primary_rows].max(axis=(1, 2)))
]
actual = validation['y'][example_index].reshape(-1)
point = selected_predictions['point'][example_index].reshape(-1)
risk = selected_predictions['peak_f1'][example_index].reshape(-1)
hours = np.arange(1, 97) / 4
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(hours, actual, label='실제')
axes[0].plot(
    hours, validation['baseline_w1'][example_index].reshape(-1),
    label='전주 기준', alpha=0.6,
)
axes[0].plot(hours, point, label='수치예측 (MAE)')
axes[0].axhline(
    PEAK_THRESHOLD, color='red', linestyle='--', label='피크 기준'
)
axes[0].set_title(
    f'primary 최고 최대전력 사례 / origin '
    f'{meta_validation.iloc[example_index]["origin"]}'
)
axes[0].legend()
axes[1].plot(hours, actual, label='실제')
axes[1].plot(hours, risk, label='피크경보 분위수 예측')
axes[1].fill_between(
    hours, 0, 1, where=risk >= PEAK_THRESHOLD,
    transform=axes[1].get_xaxis_transform(),
    alpha=0.15, color='red', label='경보',
)
axes[1].axhline(PEAK_THRESHOLD, color='red', linestyle='--')
axes[1].set_xlabel('예측 구간 내 시간')
axes[1].legend()
fig.tight_layout()
fig.savefig(
    OUT / 'selected_example_validation.png',
    dpi=160, bbox_inches='tight',
)
plt.show()


## 7. 결과와 모델 저장

역할별 전략, 전문 모델, 피처 순서, 스케일러, 손실 설정을 함께 저장합니다. 기존 GRU 결과와 섞이지 않도록 새 폴더에 저장하며, 저장한 가중치와 스케일러의 형태를 다시 확인합니다.


In [ ]:
def save_routed(role, routed, directory):
    directory.mkdir(parents=True, exist_ok=True)
    manifest = {
        'role': role,
        'strategy': routed['strategy'],
        'selection_metric': 'MAE' if role == 'point' else 'F1',
        'selection_period': '2021-07 primary_eligible',
        'test_used': False,
        'seed': SEED,
        'peak_threshold': PEAK_THRESHOLD,
        'production_split': production_split,
        'source_sha256': prep_config['source_sha256'],
        'recent_features': RECENT_NAMES,
        'routing': (
            '각 예측 대상 시간의 생산계획·주말·공휴일로 분기; '
            '전문 모델은 24시간 전체 입력을 읽고 담당 위치의 손실만 학습'
        ),
        'fallbacks': routed['fallbacks'],
        'groups': {},
    }
    for group, run in routed['models'].items():
        stem = f'{role}_{group}'
        weight_path = directory / f'{stem}.pt'
        scaler_path = directory / f'{stem}_scalers.npz'
        torch.save(run['state_dict'], weight_path)
        np.savez_compressed(
            scaler_path,
            x_mean=run['x_mean'], x_std=run['x_std'],
            target_mean=run['target_mean'], target_std=run['target_std'],
        )
        run['history'].to_csv(
            directory / f'{stem}_history.csv',
            index=False, encoding='utf-8-sig',
        )
        manifest['groups'][group] = {
            'sequence_features': run['names'],
            'params': run['params'],
            'architecture': run['architecture'],
            'objective': run['objective'],
            'alpha': run['alpha'],
            'best_epoch': run['best_epoch'],
            'best_valid_loss': run['best_valid_loss'],
            'train_positions': run['train_positions'],
            'validation_positions': run['validation_positions'],
            'weights': weight_path.name,
            'scalers': scaler_path.name,
        }
        restored = torch.load(
            weight_path, map_location='cpu', weights_only=True
        )
        assert set(restored) == set(run['state_dict'])
        with np.load(scaler_path) as saved:
            assert saved['x_mean'].shape == run['x_mean'].shape
            assert saved['x_std'].shape == run['x_std'].shape
    (directory / f'{role}_selection.json').write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    return manifest

selection_manifests = {
    role: save_routed(role, routed, OUT / 'selected_models')
    for role, routed in selected_routed.items()
}
np.savez_compressed(
    OUT / 'selected_predictions_validation.npz',
    sample_id=validation['sample_id'],
    primary_eligible=primary_rows,
    history_clean_eligible=strict_rows,
    y=validation['y'].reshape(-1, 96),
    baseline=validation['baseline_w1'].reshape(-1, 96),
    **{
        role: prediction.reshape(-1, 96)
        for role, prediction in selected_predictions.items()
    },
)
progress_results.to_csv(
    OUT / 'progress_validation.csv',
    index=False, encoding='utf-8-sig',
)
print('역할별 모델·스케일러·예측 저장 확인:', OUT.resolve())


## 결론

수치예측과 피크경보의 선택 결과를 분리해서 출력합니다. 입력과 평가 표본은 01-2 기준으로 고정했으며, GRU는 1주 전 기준선의 잔차와 24시간 시퀀스를 학습합니다.

결론 뒤의 추가실험에서는 역할별로 입력을 단순화하고, 24시간 최대전력 한 개를 직접 예측하는 GRU를 비교합니다.


In [ ]:
for role in ['point', 'peak_f1']:
    score = evaluate(
        validation['y'][primary_rows],
        selected_predictions[role][primary_rows],
    )
    run = tuned_selected[role]
    print(
        f'{role}: 전략={selected_strategy[role]}, '
        f'입력={len(run["names"]) + len(RECENT_NAMES)}채널'
    )
    if role == 'point':
        print(f'  MAE={score["mae"]:.4f}, RMSE={score["rmse"]:.4f}')
    else:
        print(
            f'  Precision={score["peak_precision"]:.4f}, '
            f'Recall={score["peak_recall"]:.4f}, '
            f'F1={score["peak_f1"]:.4f}, '
            f'FP={score["false_positive"]}, '
            f'FN={score["false_negative"]}'
        )
print('7월 검증 선택 후보이며 최종 테스트 성능은 아직 평가하지 않았습니다.')


## 추가실험

### 추가실험 A. 입력 단순화

01-2의 전체 입력과 주간 전력만 남긴 입력을 역할별 최종 전략·파라미터에 적용합니다.

| 후보 | 1·2주 전 정보 | 최근 이력 |
|---|---|---|
| reference | 01-2의 전체 입력 | 직전 시간 전력 4개 |
| weekly_power_only | 주간 전력만 유지 | 직전 시간 전력 4개 |

현재·미래 생산계획과 달력은 유지합니다. GRU 잔차 기준선인 1주 전 동일 시점 전력도 두 후보에서 유지합니다.

수치예측은 MAE, 피크경보는 F1으로 각각 선택합니다. history_clean 결과는 민감도 확인에만 사용합니다.


In [ ]:
EXTRA_OUT = OUT / 'additional_experiments'
EXTRA_OUT.mkdir(parents=True, exist_ok=True)
extra_variants = {
    'reference': False,
    'weekly_power_only': True,
}

def simplified_names(reference_names, weekly_only):
    names = list(reference_names)
    if weekly_only:
        names = [
            name for name in names
            if not name.startswith(('w1_', 'w2_'))
            or name.startswith(('w1_power_', 'w2_power_'))
        ]
    return names

extra_features, extra_routed, extra_predictions = {}, {}, {}
extra_rows = []
for role, reference in tuned_selected.items():
    seen = {}
    for variant, switches in extra_variants.items():
        names = simplified_names(reference['names'], switches)
        signature = tuple(names)
        extra_features[(role, variant)] = names
        same_as = seen.get(signature, '')
        if same_as:
            routed = extra_routed[(role, same_as)]
        elif variant == 'reference':
            routed = selected_routed[role]
        else:
            unified = fit_gru(
                names, role, reference['params'],
                reference['objective'], reference['alpha'],
                tag=f'ablation/{role}/{variant}/unified',
            )
            routed = fit_routed(
                role, unified, selected_strategy[role],
                tag_prefix=f'ablation/{variant}',
            )
        seen.setdefault(signature, variant)
        extra_routed[(role, variant)] = routed
        extra_predictions[(role, variant)] = routed['prediction']
        for subset, rows in [
            ('primary', primary_rows),
            ('history_clean', strict_rows),
        ]:
            extra_rows.append({
                'role': role, 'variant': variant,
                'subset': subset,
                'strategy': selected_strategy[role],
                'input_channels': len(names) + len(RECENT_NAMES),
                'same_input_as': same_as,
                **evaluate(
                    validation['y'][rows],
                    routed['prediction'][rows],
                ),
            })

extra_results = pd.DataFrame(extra_rows)
extra_results.to_csv(
    EXTRA_OUT / 'input_ablation_metrics.csv',
    index=False, encoding='utf-8-sig',
)
extra_selected = {}
for role in ['point', 'peak_f1']:
    table = extra_results.loc[
        extra_results['role'].eq(role) &
        extra_results['subset'].eq('primary')
    ].copy()
    extra_selected[role] = select_id(table, role, 'variant')
    columns = [
        'variant', 'strategy', 'input_channels', 'same_input_as'
    ]
    columns += (
        ['mae', 'rmse'] if role == 'point'
        else [
            'peak_precision', 'peak_recall', 'peak_f1',
            'false_positive', 'false_negative',
        ]
    )
    print('\n수치예측 (MAE 기준)' if role == 'point'
          else '\n피크경보 (F1 기준)')
    display(table[columns].round(4))
print('역할별 입력 선택:', extra_selected)
print('history_clean은 후보 선택에 사용하지 않습니다.')
display(extra_results.loc[
    extra_results['subset'].eq('history_clean'),
    [
        'role', 'variant', 'mae',
        'peak_precision', 'peak_recall', 'peak_f1',
    ],
].round(4))


### 추가실험 A 결과 시각화 및 저장

왼쪽은 수치예측 모델의 MAE, 가운데는 피크경보 모델의 F1, 오른쪽은 선택된 피크경보 모델의 혼동행렬입니다. 서로 다른 역할의 수치를 하나의 모델 성능처럼 합치지 않습니다.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
for ax, role, metric, title in [
    (axes[0], 'point', 'mae', '수치예측 MAE (낮을수록 좋음)'),
    (axes[1], 'peak_f1', 'peak_f1', '피크경보 F1 (높을수록 좋음)'),
]:
    table = extra_results.loc[
        extra_results['role'].eq(role) &
        extra_results['subset'].eq('primary')
    ]
    colors = [
        'tab:orange' if value == extra_selected[role] else 'tab:blue'
        for value in table['variant']
    ]
    ax.bar(table['variant'], table[metric], color=colors)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=35)
    for index, value in enumerate(table[metric]):
        ax.text(index, value, f'{value:.4f}', ha='center', va='bottom')

risk_prediction = extra_predictions[
    ('peak_f1', extra_selected['peak_f1'])
]
risk_score = evaluate(
    validation['y'][primary_rows],
    risk_prediction[primary_rows],
)
matrix = np.array([
    [risk_score['true_negative'], risk_score['false_positive']],
    [risk_score['false_negative'], risk_score['true_positive']],
])
axes[2].imshow(matrix, cmap='Blues')
for (row, col), value in np.ndenumerate(matrix):
    axes[2].text(
        col, row, f'{value:,}\n({value / matrix.sum():.2%})',
        ha='center', va='center',
        color='white' if value > matrix.max() / 2 else 'black',
    )
axes[2].set(
    xticks=[0, 1], yticks=[0, 1],
    xticklabels=['예측 비피크', '예측 피크'],
    yticklabels=['실제 비피크', '실제 피크'],
    xlabel='예측', ylabel='실제',
    title=(
        f'F1 선택: {extra_selected["peak_f1"]}\n'
        f'피크 기준 {PEAK_THRESHOLD:g}'
    ),
)
fig.tight_layout()
fig.savefig(
    EXTRA_OUT / 'input_ablation_validation.png',
    dpi=160, bbox_inches='tight',
)
plt.show()

extra_condition_rows = []
for (role, variant), prediction in extra_predictions.items():
    for condition, mask in conditions.items():
        extra_condition_rows.append({
            'role': role, 'variant': variant,
            'condition': condition,
            **evaluate(
                validation['y'], prediction,
                mask & primary_rows[:, None, None],
            ),
        })
pd.DataFrame(extra_condition_rows).to_csv(
    EXTRA_OUT / 'input_ablation_conditions.csv',
    index=False, encoding='utf-8-sig',
)
np.savez_compressed(
    EXTRA_OUT / 'input_ablation_predictions.npz',
    sample_id=validation['sample_id'],
    primary_eligible=primary_rows,
    history_clean_eligible=strict_rows,
    y=validation['y'].reshape(-1, 96),
    **{
        f'{role}__{variant}': prediction.reshape(-1, 96)
        for (role, variant), prediction in extra_predictions.items()
    },
)
extra_manifests = {
    role: save_routed(
        role,
        extra_routed[(role, extra_selected[role])],
        EXTRA_OUT / 'selected_models',
    )
    for role in ['point', 'peak_f1']
}
(EXTRA_OUT / 'input_ablation_selection.json').write_text(
    json.dumps({
        'selection_period': '2021-07 primary_eligible',
        'test_used': False,
        'selected_variants': extra_selected,
        'manifests': extra_manifests,
    }, ensure_ascii=False, indent=2),
    encoding='utf-8',
)


### 추가실험 B. 24시간 최대전력 직접 예측

15분별 수치예측 96개의 최댓값과, 향후 24시간 실제 최대전력 한 개를 직접 학습한 GRU를 비교합니다. 추가실험 A에서 선택한 수치예측 입력과 구조를 사용하고 통합 모델 하나를 학습합니다.

직접 최대전력 모델은 최대값의 크기만 출력하므로 발생 시각, 15분별 F1, 혼동행렬을 계산할 수 없습니다. 기존 수치예측·피크경보 모델을 대체하지 않으며 최대값 MAE·RMSE·편향·과소예측 비율로만 평가합니다.


In [ ]:
maximum_names = extra_features[
    ('point', extra_selected['point'])
]
maximum_params = tuned_selected['point']['params']
xtr_raw = make_input(train, maximum_names)
xva_raw = make_input(validation, maximum_names)
maximum_x_mean = xtr_raw.mean(axis=(0, 1), keepdims=True)
maximum_x_std = np.maximum(
    xtr_raw.std(axis=(0, 1), keepdims=True), 1e-6
)
xtr = ((xtr_raw - maximum_x_mean) / maximum_x_std).astype(np.float32)
xva = ((xva_raw - maximum_x_mean) / maximum_x_std).astype(np.float32)
maximum_y_train = train['y'].reshape(len(train['y']), -1).max(axis=1)
maximum_y_valid = validation['y'].reshape(
    len(validation['y']), -1
).max(axis=1)
maximum_target_mean = float(maximum_y_train.mean())
maximum_target_std = max(float(maximum_y_train.std()), 1e-6)
scaled_y_train = (
    (maximum_y_train - maximum_target_mean) / maximum_target_std
).astype(np.float32)
scaled_y_valid = (
    (maximum_y_valid - maximum_target_mean) / maximum_target_std
).astype(np.float32)

torch.manual_seed(SEED)
np.random.seed(SEED)
maximum_architecture = {
    'input_features': xtr.shape[-1],
    **{
        key: maximum_params[key]
        for key in [
            'hidden_size', 'num_layers',
            'dropout', 'bidirectional',
        ]
    },
    'scalar_output': True,
}
maximum_model = GRURegressor(**maximum_architecture).to(device)
maximum_optimizer = torch.optim.AdamW(
    maximum_model.parameters(),
    lr=maximum_params['learning_rate'],
    weight_decay=maximum_params['weight_decay'],
)
maximum_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(xtr),
        torch.from_numpy(scaled_y_train),
    ),
    batch_size=maximum_params['batch_size'],
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
valid_x = torch.from_numpy(xva).to(device)
valid_target = torch.from_numpy(scaled_y_valid).to(device)
valid_mask = torch.from_numpy(primary_rows).to(device)

best_loss, best_epoch, best_state, stale = np.inf, 0, None, 0
maximum_history = []
for epoch in range(1, maximum_params['max_epochs'] + 1):
    maximum_model.train()
    train_sum, train_count = 0.0, 0
    for batch_x, batch_y in maximum_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        maximum_optimizer.zero_grad()
        loss = (maximum_model(batch_x) - batch_y).square().mean()
        loss.backward()
        nn.utils.clip_grad_norm_(maximum_model.parameters(), 1.0)
        maximum_optimizer.step()
        train_sum += float(loss.detach()) * len(batch_x)
        train_count += len(batch_x)

    maximum_model.eval()
    with torch.no_grad():
        valid_loss = float(
            (maximum_model(valid_x)[valid_mask] -
             valid_target[valid_mask]).square().mean()
        )
    maximum_history.append({
        'epoch': epoch,
        'train_loss': train_sum / train_count,
        'valid_loss': valid_loss,
    })
    if valid_loss < best_loss - 1e-6:
        best_loss, best_epoch, stale = valid_loss, epoch, 0
        best_state = {
            name: tensor.detach().cpu().clone()
            for name, tensor in maximum_model.state_dict().items()
        }
    else:
        stale += 1
        if stale >= maximum_params['patience']:
            break

maximum_model.load_state_dict(best_state)
maximum_model.eval()
parts = []
with torch.no_grad():
    for start in range(0, len(xva), maximum_params['batch_size']):
        parts.append(maximum_model(
            torch.from_numpy(
                xva[start:start + maximum_params['batch_size']]
            ).to(device)
        ).cpu().numpy())
direct_maximum = np.maximum(
    np.concatenate(parts) * maximum_target_std +
    maximum_target_mean,
    0,
).astype(np.float32)

maximum_predictions = {
    'previous_week_max': validation['baseline_w1'].reshape(
        len(validation['y']), -1
    ).max(axis=1),
    'selected_point_curve_max': selected_predictions['point'].reshape(
        len(validation['y']), -1
    ).max(axis=1),
    'ablation_selected_curve_max': extra_predictions[
        ('point', extra_selected['point'])
    ].reshape(len(validation['y']), -1).max(axis=1),
    'direct_maximum_gru': direct_maximum,
}
maximum_rows = []
for subset, rows in [
    ('primary', primary_rows),
    ('history_clean', strict_rows),
]:
    for model_name, prediction in maximum_predictions.items():
        maximum_rows.append({
            'subset': subset, 'model': model_name,
            **maximum_metrics(
                maximum_y_valid[rows], prediction[rows]
            ),
        })
direct_maximum_results = pd.DataFrame(maximum_rows)
display(direct_maximum_results.round(4))
print(
    '편향 = 예측 최대값 - 실제 최대값. '
    '음수이면 최대전력을 낮게 예측한 것입니다.'
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
table = direct_maximum_results.loc[
    direct_maximum_results['subset'].eq('primary')
]
axes[0].bar(table['model'], table['maximum_mae'])
axes[0].set_title('24시간 최대전력 MAE')
axes[0].tick_params(axis='x', rotation=30)
axes[1].scatter(
    maximum_y_valid[primary_rows],
    direct_maximum[primary_rows],
    alpha=0.3, s=14,
)
bounds = [
    min(
        maximum_y_valid[primary_rows].min(),
        direct_maximum[primary_rows].min(),
    ),
    max(
        maximum_y_valid[primary_rows].max(),
        direct_maximum[primary_rows].max(),
    ),
]
axes[1].plot(bounds, bounds, 'k--')
axes[1].set(
    xlabel='실제 최대전력',
    ylabel='직접 예측 최대전력',
    title='대각선 아래: 최대전력 과소예측',
)
fig.tight_layout()
fig.savefig(
    EXTRA_OUT / 'direct_maximum_validation.png',
    dpi=160, bbox_inches='tight',
)
plt.show()

direct_maximum_results.to_csv(
    EXTRA_OUT / 'maximum_metrics.csv',
    index=False, encoding='utf-8-sig',
)
pd.DataFrame(maximum_history).to_csv(
    EXTRA_OUT / 'direct_maximum_history.csv',
    index=False, encoding='utf-8-sig',
)
torch.save(
    best_state, EXTRA_OUT / 'direct_maximum_gru.pt'
)
np.savez_compressed(
    EXTRA_OUT / 'direct_maximum_scalers.npz',
    x_mean=maximum_x_mean, x_std=maximum_x_std,
    target_mean=maximum_target_mean,
    target_std=maximum_target_std,
)
np.savez_compressed(
    EXTRA_OUT / 'maximum_predictions.npz',
    sample_id=validation['sample_id'],
    primary_eligible=primary_rows,
    history_clean_eligible=strict_rows,
    y_maximum=maximum_y_valid,
    **maximum_predictions,
)
(EXTRA_OUT / 'direct_maximum_config.json').write_text(
    json.dumps({
        'sequence_features': maximum_names,
        'recent_features': RECENT_NAMES,
        'architecture': maximum_architecture,
        'params': maximum_params,
        'best_epoch': best_epoch,
        'best_valid_loss': best_loss,
        'target': 'maximum power over next 24 hours (96 quarters)',
        'selection_period': '2021-07 primary_eligible',
        'test_used': False,
    }, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print('추가실험 저장 위치:', EXTRA_OUT.resolve())


### 추가실험 결과 판단 기준

- 입력 단순화는 수치예측의 MAE 감소와 피크경보의 F1 증가를 각각 판단합니다. F1과 함께 Precision·Recall·오탐·미탐을 확인합니다.
- 실제 입력이 같은 후보는 독립 실험으로 해석하지 않습니다.
- 직접 최대전력 GRU는 전주 최대값, 최종 수치예측 곡선의 최대값과 비교합니다. 최대값 MAE와 음의 편향이 함께 줄어야 의미가 있습니다.
- 모든 선택은 7월 검증 결과입니다. 기존 최종 모델을 자동 교체하지 않으며, 월별 롤링 검증에서 개선의 반복 여부를 확인한 뒤 채택합니다.


## 10. 02 LGBM과 동일 조건의 GRU 비교 실험

02와 동일한 공식 30개 입력·개선 47개 입력, 월요일별 재학습, 이례적 휴무 가중치, 3단 규칙, 가동 시간 잔차 학습, 휴무일 대표값, 7월 평가 60,192칸과 피크 기준 176을 적용합니다.

피크경보는 분위수 0.75·0.80·0.90과 입력·휴무 가중치 효과를 비교합니다. 수치예측은 전체 시간 학습, 가동 시간 학습, 휴무일 처리 효과를 비교합니다. 지원 코드는 `03_2_aligned_gru.py`, 결과는 `results/gru_24h_team_preprocessing/aligned_with_team_lgbm/`에 저장됩니다.


In [ ]:
aligned_script = Path('03_2_aligned_gru.py')
if not aligned_script.exists():
    aligned_script = ROOT / 'real_final/03_2_aligned_gru.py'
exec(aligned_script.read_text(encoding='utf-8'), globals())


In [ ]:
# ============================================================
# 11. 8~9월 동일 조건 개발 검증 및 최종 GRU 선정
# ============================================================

# 02와 동일하게 8~9월에도 매주 월요일 이전까지 정답이 확정된
# 데이터만 사용하여 다시 학습합니다.
AW_TEST_WEEKS = pd.date_range(
    "2021-08-02", "2021-09-13", freq="7D"
)

assert len(W.ev_test) == 1057
assert W.y[W.ev_test].size == 1057 * 96
assert W.meta.loc[W.ev_test, "origin"].dt.month.isin([8, 9]).all()


def aw_predict_additional_weeks(
    initial_prediction,
    candidate,
    inputs,
    target,
    objective,
    weeks,
    alpha=0.5,
    training_cells=None,
    use_off_weight=True,
):
    """
    기존 7월 예측을 유지하고 8~9월 주별 예측을 이어서 생성합니다.
    각 주 시작 전에 정답이 확정된 창만 학습에 사용합니다.
    """
    prediction = initial_prediction.copy()

    for week_start in weeks:
        train_rows = W.meta["target_end"].le(week_start).to_numpy()
        predict_rows = (
            W.meta["origin"].ge(week_start)
            & W.meta["origin"].lt(
                week_start + pd.Timedelta(days=7)
            )
        ).to_numpy()

        if not predict_rows.any():
            continue

        assert W.meta.loc[
            train_rows, "target_end"
        ].max() <= week_start

        weights = np.ones_like(
            target[train_rows], dtype=np.float32
        )

        known_off_windows = aw_known_off_windows(week_start)

        if use_off_weight:
            weights *= np.where(
                known_off_windows[train_rows, None, None],
                0.1,
                1.0,
            ).astype(np.float32)

        if training_cells is not None:
            weights *= training_cells[train_rows].astype(
                np.float32
            )

        started = time.time()

        week_prediction, history = aw_fit_predict(
            inputs[train_rows],
            target[train_rows],
            inputs[predict_rows],
            objective,
            alpha=alpha,
            cell_weight=weights,
        )

        prediction[predict_rows] = week_prediction

        AW_SCHEDULE.append({
            "candidate": candidate,
            "week_start": str(week_start.date()),
            "training_windows": int(train_rows.sum()),
            "prediction_windows": int(predict_rows.sum()),
            "downweighted_windows": int(
                (known_off_windows & train_rows).sum()
                if use_off_weight else 0
            ),
            "epochs": AW_EPOCHS,
            "last_train_loss": history[-1],
            "seconds": time.time() - started,
        })

        print(
            candidate,
            week_start.date(),
            f"학습 {int(train_rows.sum()):,}창",
            f"예측 {int(predict_rows.sum()):,}창",
        )

    return prediction


# ------------------------------------------------------------
# 1. 현재 Q1 대응 GRU: 1~7월 학습 후 8~9월 전체 예측
# ------------------------------------------------------------

aw_static_test_q090, aw_static_test_history = (
    aw_fit_predict(
        AW_X30[W.tr_test],
        W.y3[W.tr_test],
        AW_X30[W.ev_test],
        "quantile",
        alpha=0.90,
    )
)

aw_static_test_q090 = np.clip(
    aw_static_test_q090.reshape(-1, 96),
    0,
    None,
)

AW_SCHEDULE.append({
    "candidate": "gru_current_static_30_q090",
    "week_start": "2021-08-01",
    "training_windows": int(W.tr_test.sum()),
    "prediction_windows": int(len(W.ev_test)),
    "downweighted_windows": 0,
    "epochs": AW_EPOCHS,
    "last_train_loss": aw_static_test_history[-1],
    "seconds": np.nan,
})


# ------------------------------------------------------------
# 2. 개선 Q1 대응 GRU: 8~9월 주별 재학습
# ------------------------------------------------------------

AW_PEAK_PREDICTIONS_ALL = {}

for candidate, inputs, alpha, use_off_weight in AW_PEAK_SPECS:
    # 기존 배열에는 5월 31일~8월 1일 예측이 들어 있습니다.
    initial = AW_PEAK_PREDICTIONS[candidate].reshape(
        W.N, 24, 4
    )

    prediction = aw_predict_additional_weeks(
        initial_prediction=initial,
        candidate=candidate,
        inputs=inputs,
        target=W.y3,
        objective="quantile",
        weeks=AW_TEST_WEEKS,
        alpha=alpha,
        training_cells=None,
        use_off_weight=use_off_weight,
    )

    prediction = np.clip(
        prediction.reshape(W.N, 96),
        0,
        None,
    )

    assert np.isfinite(
        prediction[W.ev_val]
    ).all()

    assert np.isfinite(
        prediction[W.ev_test]
    ).all()

    AW_PEAK_PREDICTIONS_ALL[candidate] = prediction


# ------------------------------------------------------------
# 3. 수치예측 GRU: 8~9월 주별 잔차 재학습
# ------------------------------------------------------------

aw_curve_all_full = aw_predict_additional_weeks(
    initial_prediction=(
        aw_curve_all - AW_RULE
    ).reshape(W.N, 24, 4),
    candidate="gru_curve_residual_all_cells",
    inputs=AW_X47,
    target=AW_RESIDUAL_TARGET,
    objective="l1",
    weeks=AW_TEST_WEEKS,
    training_cells=None,
    use_off_weight=False,
)

aw_curve_all_full = np.clip(
    aw_curve_all_full.reshape(W.N, 96) + AW_RULE,
    0,
    None,
)

aw_curve_on_full = aw_predict_additional_weeks(
    initial_prediction=(
        aw_curve_on - AW_RULE
    ).reshape(W.N, 24, 4),
    candidate="gru_curve_residual_operating_cells",
    inputs=AW_X47,
    target=AW_RESIDUAL_TARGET,
    objective="l1",
    weeks=AW_TEST_WEEKS,
    training_cells=~AW_GATE,
    use_off_weight=False,
)

aw_curve_on_full = np.clip(
    aw_curve_on_full.reshape(W.N, 96) + AW_RULE,
    0,
    None,
)

aw_curve_on_rule_off_full = np.where(
    aw_gate_flat,
    AW_RULE,
    aw_curve_on_full,
).astype(np.float32)

aw_curve_final_full = np.where(
    aw_gate_flat & np.isfinite(AW_IDLE),
    AW_IDLE,
    aw_curve_on_rule_off_full,
).astype(np.float32)

assert np.isfinite(
    aw_curve_all_full[W.ev_test]
).all()

assert np.isfinite(
    aw_curve_final_full[W.ev_test]
).all()


# ------------------------------------------------------------
# 4. 7월·8~9월·전체 기간의 피크경보 성능 비교
# ------------------------------------------------------------

AW_STATIC_FULL = np.full(
    (W.N, 96),
    np.nan,
    dtype=np.float32,
)

AW_STATIC_FULL[W.ev_val] = aw_static_q090
AW_STATIC_FULL[W.ev_test] = aw_static_test_q090

aw_peak_candidates_all = {
    "gru_current_static_30_q090": AW_STATIC_FULL,
    **AW_PEAK_PREDICTIONS_ALL,
}

aw_periods = {
    "validation_7월": W.ev_val,
    "development_8_9월": W.ev_test,
    "pooled_7_9월": np.concatenate([
        W.ev_val,
        W.ev_test,
    ]),
}

aw_peak_period_rows = []

for period, rows in aw_periods.items():
    for candidate, prediction in (
        aw_peak_candidates_all.items()
    ):
        aw_peak_period_rows.append({
            "period": period,
            "candidate": candidate,
            **aw_metrics(
                W.y[rows],
                prediction[rows],
                AW_PEAK,
            ),
        })

aw_peak_period_results = pd.DataFrame(
    aw_peak_period_rows
)

display(
    aw_peak_period_results[[
        "period",
        "candidate",
        "precision",
        "recall",
        "f1",
        "f2",
        "fp",
        "fn",
        "mae",
    ]].round(4)
)


# 전체 7~9월 F1을 기준으로 최종 피크경보 GRU를 선택합니다.
aw_peak_pooled = (
    aw_peak_period_results.loc[
        aw_peak_period_results[
            "period"
        ].eq("pooled_7_9월")
    ]
    .copy()
)

# 전체 F1이 같으면 두 기간 중 낮은 F1이 더 높은 모델,
# 그다음 정밀도가 높은 모델을 선택합니다.
aw_period_f1 = (
    aw_peak_period_results.loc[
        aw_peak_period_results["period"].isin([
            "validation_7월",
            "development_8_9월",
        ])
    ]
    .pivot(
        index="candidate",
        columns="period",
        values="f1",
    )
)

aw_period_f1["minimum_period_f1"] = (
    aw_period_f1.min(axis=1)
)

aw_peak_pooled = aw_peak_pooled.merge(
    aw_period_f1[["minimum_period_f1"]],
    left_on="candidate",
    right_index=True,
    how="left",
)

aw_selected_peak_row = (
    aw_peak_pooled.sort_values(
        [
            "f1",
            "minimum_period_f1",
            "precision",
            "recall",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
        kind="stable",
    )
    .iloc[0]
)

AW_SELECTED_PEAK_MODEL = (
    aw_selected_peak_row["candidate"]
)

AW_SELECTED_PEAK_PREDICTION = (
    aw_peak_candidates_all[
        AW_SELECTED_PEAK_MODEL
    ]
)

print(
    "최종 피크경보 GRU:",
    AW_SELECTED_PEAK_MODEL,
)

print(
    "7~9월 통합 F1:",
    round(
        float(aw_selected_peak_row["f1"]),
        4,
    ),
)


# ------------------------------------------------------------
# 5. 7월·8~9월·전체 기간의 수치예측 성능 비교
# ------------------------------------------------------------

aw_curve_candidates_all = {
    "previous_week_baseline": W.base,
    "rule3_oof": AW_RULE,
    "gru_residual_all_cells": (
        aw_curve_all_full
    ),
    "gru_residual_operating_cells_rule_off": (
        aw_curve_on_rule_off_full
    ),
    "gru_residual_operating_cells_idle_profile": (
        aw_curve_final_full
    ),
}

aw_curve_period_rows = []

for period, rows in aw_periods.items():
    for candidate, prediction in (
        aw_curve_candidates_all.items()
    ):
        score = aw_metrics(
            W.y[rows],
            prediction[rows],
            AW_PEAK,
        )

        aw_curve_period_rows.append({
            "period": period,
            "candidate": candidate,
            "mae": score["mae"],
            "rmse": score["rmse"],
            "bias": score["bias"],
        })

aw_curve_period_results = pd.DataFrame(
    aw_curve_period_rows
)

display(
    aw_curve_period_results.round(4)
)


# GRU 수치예측 후보 중 7~9월 통합 MAE가 가장 낮은 모델을 선택합니다.
aw_curve_pooled = (
    aw_curve_period_results.loc[
        aw_curve_period_results[
            "period"
        ].eq("pooled_7_9월")
        & aw_curve_period_results[
            "candidate"
        ].str.startswith("gru_")
    ]
    .copy()
)

aw_curve_period_mae = (
    aw_curve_period_results.loc[
        aw_curve_period_results["period"].isin([
            "validation_7월",
            "development_8_9월",
        ])
        & aw_curve_period_results[
            "candidate"
        ].str.startswith("gru_")
    ]
    .pivot(
        index="candidate",
        columns="period",
        values="mae",
    )
)

aw_curve_period_mae["maximum_period_mae"] = (
    aw_curve_period_mae.max(axis=1)
)

aw_curve_pooled = aw_curve_pooled.merge(
    aw_curve_period_mae[
        ["maximum_period_mae"]
    ],
    left_on="candidate",
    right_index=True,
    how="left",
)

aw_selected_curve_row = (
    aw_curve_pooled.sort_values(
        [
            "mae",
            "maximum_period_mae",
            "rmse",
        ],
        ascending=[
            True,
            True,
            True,
        ],
        kind="stable",
    )
    .iloc[0]
)

AW_SELECTED_CURVE_MODEL = (
    aw_selected_curve_row["candidate"]
)

AW_SELECTED_CURVE_PREDICTION = (
    aw_curve_candidates_all[
        AW_SELECTED_CURVE_MODEL
    ]
)

print(
    "최종 수치예측 GRU:",
    AW_SELECTED_CURVE_MODEL,
)

print(
    "7~9월 통합 MAE:",
    round(
        float(aw_selected_curve_row["mae"]),
        4,
    ),
)


# ------------------------------------------------------------
# 6. 결과 저장
# ------------------------------------------------------------

aw_peak_period_results.to_csv(
    AW_OUT / "aligned_peak_metrics_all_periods.csv",
    index=False,
    encoding="utf-8-sig",
)

aw_curve_period_results.to_csv(
    AW_OUT / "aligned_curve_metrics_all_periods.csv",
    index=False,
    encoding="utf-8-sig",
)

pd.DataFrame(AW_SCHEDULE).to_csv(
    AW_OUT / "training_schedule_all_periods.csv",
    index=False,
    encoding="utf-8-sig",
)


def aw_prediction_frame(rows, period):
    origins = W.meta.loc[
        rows,
        "origin",
    ].to_numpy()

    steps = np.tile(
        np.arange(96),
        len(rows),
    )

    origin_column = np.repeat(
        origins,
        96,
    )

    target_time = (
        pd.to_datetime(origin_column)
        + pd.to_timedelta(
            steps // 4,
            unit="h",
        )
        + pd.to_timedelta(
            15 * (steps % 4 + 1),
            unit="m",
        )
    )

    frame = pd.DataFrame({
        "period": period,
        "기준시각": origin_column,
        "예측대상시각": target_time,
        "예측칸": steps,
        "실제전력": W.y[rows].reshape(-1),
        "baseline_w1": W.base[
            rows
        ].reshape(-1),
        "rule3_oof": AW_RULE[
            rows
        ].reshape(-1),
    })

    for candidate, prediction in (
        aw_peak_candidates_all.items()
    ):
        frame[candidate] = prediction[
            rows
        ].reshape(-1)

    for candidate, prediction in (
        aw_curve_candidates_all.items()
    ):
        if candidate.startswith("gru_"):
            frame[candidate] = prediction[
                rows
            ].reshape(-1)

    frame["selected_peak_gru"] = (
        AW_SELECTED_PEAK_PREDICTION[
            rows
        ].reshape(-1)
    )

    frame["selected_curve_gru"] = (
        AW_SELECTED_CURVE_PREDICTION[
            rows
        ].reshape(-1)
    )

    frame["peak_threshold"] = AW_PEAK

    frame["actual_peak"] = (
        frame["실제전력"] >= AW_PEAK
    ).astype(np.int8)

    frame["selected_gru_alarm"] = (
        frame["selected_peak_gru"]
        >= AW_PEAK
    ).astype(np.int8)

    return frame


aw_predictions_7월 = aw_prediction_frame(
    W.ev_val,
    "validation_7월",
)

aw_predictions_8_9월 = aw_prediction_frame(
    W.ev_test,
    "development_8_9월",
)

aw_predictions_all = pd.concat(
    [
        aw_predictions_7월,
        aw_predictions_8_9월,
    ],
    ignore_index=True,
)

aw_predictions_7월.to_csv(
    AW_OUT / "aligned_predictions_validation_7월.csv",
    index=False,
    encoding="utf-8-sig",
)

aw_predictions_8_9월.to_csv(
    AW_OUT / "aligned_predictions_development_8_9월.csv",
    index=False,
    encoding="utf-8-sig",
)

aw_predictions_all.to_csv(
    AW_OUT / "aligned_predictions_all_7_9월.csv",
    index=False,
    encoding="utf-8-sig",
)


aw_final_selection = {
    "selection_period": (
        "2021-07 validation + "
        "2021-08~09 development validation"
    ),
    "independent_test_remaining": False,
    "actual_peak_threshold": AW_PEAK,
    "peak_selection_metric": (
        "pooled F1; tie: minimum period F1, "
        "precision, recall"
    ),
    "selected_peak_gru": (
        AW_SELECTED_PEAK_MODEL
    ),
    "selected_peak_metrics_pooled": {
        key: (
            int(aw_selected_peak_row[key])
            if key in [
                "observations",
                "tp",
                "fp",
                "fn",
                "tn",
            ]
            else float(
                aw_selected_peak_row[key]
            )
        )
        for key in [
            "observations",
            "mae",
            "rmse",
            "bias",
            "precision",
            "recall",
            "f1",
            "f2",
            "tp",
            "fp",
            "fn",
            "tn",
        ]
    },
    "curve_selection_metric": (
        "pooled MAE; tie: maximum period MAE, RMSE"
    ),
    "selected_curve_gru": (
        AW_SELECTED_CURVE_MODEL
    ),
    "selected_curve_metrics_pooled": {
        "mae": float(
            aw_selected_curve_row["mae"]
        ),
        "rmse": float(
            aw_selected_curve_row["rmse"]
        ),
        "bias": float(
            aw_selected_curve_row["bias"]
        ),
    },
    "weekly_retraining_through": (
        "2021-09-13"
    ),
    "test_used_for_selection": True,
    "test_label": "development_validation",
}

(
    AW_OUT / "final_gru_selection_7_9월.json"
).write_text(
    json.dumps(
        aw_final_selection,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "저장 완료:",
    AW_OUT.resolve(),
)